# Model recovery for the tuning-width scaling parameter

Analogous to `recovery_bias.ipynb` (Fig. S8, which shows recovery of the *shift* parameter
$\delta_{wide}$), but for the *width* scaling parameter `sd_wide_scale` of the
`LinearScalingModel` (model 15: per-voxel $\mu$, $\sigma$, amplitude and baseline; one
shared, freely fitted width-scaling factor; shift fixed at $\delta_{wide}=2$).

Simulations were generated with `simulate_data_sd.py`: per-voxel ground-truth
($\mu_{narrow}$, $\sigma_{narrow}$) pairs were sampled from the empirical model-15 fits in
NPCr (voxels with out-of-sample $R^2 > 0$), responses were simulated with a known
`sd_wide_scale` (1.0 = no widening; 1.29 = empirical group mean) under the full
(narrow 10–25, wide 10–40) or censored (both ranges 10–25) design, and model 15 was refit.

**Noise model**: i.i.d. Gaussian noise with standard deviation 0.5 added to every simulated
response; tuning-curve amplitude is 1 and baseline 0, so this corresponds to a peak SNR of 2
(matching the $\delta_{wide}$ recovery simulations).

In [ ]:
import numpy as np
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
bids_folder = Path('/data/ds-neuralpriors')
results_files = list(bids_folder.glob('simulated_recovery_sd/sd_scale_*/noise_*/design_*/iteration-*_results.csv'))

pars = pd.concat([pd.read_csv(f) for f in results_files], ignore_index=True)

pars['design'] = [f.parent.name.split('_')[-1] for f in results_files]
pars['noise'] = [float(f.parent.parent.name.split('_')[-1]) for f in results_files]
pars['gen_sd_wide_scale'] = [f.parent.parent.parent.name.split('_')[-1] for f in results_files]

pars['design'] = pars['design'].str.capitalize().map({'Full': 'Full (10-25 and 10-40)', 'Censored': 'Censored (All conditions 10-25)'})
pars.groupby(['gen_sd_wide_scale', 'design', 'noise'])['sd_wide_scale'].describe()

In [ ]:
pars_main = pars[pars['noise'] == 0.5]

g = sns.FacetGrid(pars_main, col='design', hue='gen_sd_wide_scale', margin_titles=True)
g.map(sns.histplot, 'sd_wide_scale', stat='density', common_norm=False, bins=np.linspace(0.5, 2., 50))

for gen_value, color in zip(['1.0', '1.287794'], sns.color_palette()):
    for ax in g.axes.ravel():
        ax.axvline(float(gen_value), color=color, ls='--', lw=1)

g.add_legend(title='Generative sd_wide_scale')
g.set_axis_labels('Estimated sd_wide_scale', 'Density')
g.set_titles(col_template='{col_name} design')

g.savefig('/data/ds-neuralpriors/derivatives/figures/model_recovery_sd.pdf')

## Per-voxel recovery of $\mu_{narrow}$ and $\sigma_{narrow}$

The reviewer notes that for voxels with monotonic (non-tuned) responses, a moderate width with
a mean near the upper stimulus range can trade off against a very large width with a much higher
mean. The per-voxel scatter of generative vs. recovered parameters shows whether such
degeneracies distort the estimates in practice.

In [ ]:
pervoxel_files = list(bids_folder.glob('simulated_recovery_sd/sd_scale_*/noise_0.5/design_*/iteration-*_pervoxel.csv'))

pervoxel = pd.concat([pd.read_csv(f).assign(design=f.parent.name.split('_')[-1],
                                            gen_sd_wide_scale=f.parent.parent.parent.name.split('_')[-1])
                      for f in pervoxel_files], ignore_index=True)

pervoxel['design'] = pervoxel['design'].str.capitalize().map({'Full': 'Full (10-25 and 10-40)', 'Censored': 'Censored (All conditions 10-25)'})
len(pervoxel)

In [ ]:
g = sns.FacetGrid(pervoxel, col='design', row='gen_sd_wide_scale', margin_titles=True, height=3.5)
g.map(sns.histplot, 'gen_sd_narrow', 'est_sd_narrow', bins=50)

for ax in g.axes.ravel():
    lims = [0, 3]
    ax.plot(lims, lims, color='k', ls='--', lw=1)
    ax.set(xlim=lims, ylim=lims)

g.set_axis_labels('Generative sd_narrow', 'Estimated sd_narrow')
g.set_titles(col_template='{col_name} design', row_template='Generative scale = {row_name}')

In [ ]:
g = sns.FacetGrid(pervoxel, col='design', row='gen_sd_wide_scale', margin_titles=True, height=3.5)
g.map(sns.histplot, 'gen_mu_narrow', 'est_mu_narrow', bins=50)

for ax in g.axes.ravel():
    lims = [0, 60]
    ax.plot(lims, lims, color='k', ls='--', lw=1)
    ax.set(xlim=lims, ylim=lims)

g.set_axis_labels('Generative mu_narrow', 'Estimated mu_narrow')
g.set_titles(col_template='{col_name} design', row_template='Generative scale = {row_name}')

In [ ]:
# Correlations between generative and recovered per-voxel parameters
pervoxel.groupby(['gen_sd_wide_scale', 'design']).apply(
    lambda d: pd.Series({'r(mu)': d['gen_mu_narrow'].corr(d['est_mu_narrow']),
                         'r(sd)': d['gen_sd_narrow'].corr(d['est_sd_narrow'])}))